# End-to-End RAG Pipeline Evaluation

## 1. Project Overview

**Task:** Build a complete RAG pipeline and evaluate it across **four dimensions**: retrieval quality, answer groundedness, cost, and latency — the metrics that matter in production.

**Why this matters:** Building a RAG system is easy. Knowing whether it's *good* is hard. Most tutorials show you how to retrieve and generate but never how to measure:
- Are you retrieving the **right chunks**? (retrieval quality)
- Is the LLM **making things up** vs citing the evidence? (groundedness)
- How much does each query **cost** in tokens? (cost)
- How **fast** is the end-to-end pipeline? (latency)

This notebook builds a RAG pipeline, creates a labeled evaluation dataset, and runs systematic evaluation across all four axes.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM (local, no API keys, zero token cost)
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store
- Pure Python metrics — no external eval frameworks needed

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand the **four pillars of RAG evaluation**: retrieval, groundedness, cost, latency |
| 2 | Build a **labeled evaluation dataset** with expected chunks and reference answers |
| 3 | Compute **retrieval metrics**: Hit Rate, MRR, Precision@K, Recall@K |
| 4 | Implement **LLM-as-judge groundedness scoring** at the claim level |
| 5 | Track **token budgets** and understand cost drivers |
| 6 | Profile **latency** across pipeline stages (embed, retrieve, generate) |
| 7 | Create a **RAG scorecard** that summarizes all metrics in one view |

## 3. The Four Pillars of RAG Evaluation

```
  ┌─────────────────────────────────────────────────────────────────────┐
  │                     RAG EVALUATION FRAMEWORK                       │
  ├──────────────────┬──────────────────┬───────────┬─────────────────┤
  │  RETRIEVAL       │  GROUNDEDNESS    │  COST     │  LATENCY        │
  │  QUALITY         │                  │           │                 │
  ├──────────────────┼──────────────────┼───────────┼─────────────────┤
  │ Did we find the  │ Is the answer    │ How many  │ How fast is the │
  │ right chunks?    │ supported by the │ tokens    │ pipeline?       │
  │                  │ retrieved text?  │ per query?│                 │
  ├──────────────────┼──────────────────┼───────────┼─────────────────┤
  │ • Hit Rate       │ • Claim-level    │ • Prompt  │ • Embedding     │
  │ • MRR            │   verification   │   tokens  │ • Retrieval     │
  │ • Precision@K    │ • Faithfulness   │ • Output  │ • Generation    │
  │ • Recall@K       │   score          │   tokens  │ • End-to-end    │
  │ • nDCG           │ • Hallucination  │ • Total   │ • Time-to-first │
  │                  │   detection      │   budget  │   token (TTFT)  │
  └──────────────────┴──────────────────┴───────────┴─────────────────┘
```

### Why Each Pillar Matters

| Pillar | Failure Mode | Real-World Impact |
|--------|-------------|-------------------|
| **Retrieval** | Wrong chunks → wrong answer | User gets irrelevant or misleading info |
| **Groundedness** | LLM hallucinates beyond evidence | User trusts fabricated facts |
| **Cost** | Huge context → expensive queries | Budget blown in production at scale |
| **Latency** | Slow pipeline → bad UX | Users abandon slow systems |

## 4. Pipeline Architecture

```
  Question
     │
     ▼               ┌──────────┐
  Embed query ──────→│ LATENCY  │ t_embed
     │               │ TRACKER  │
     ▼               │          │
  Vector search ────→│          │ t_retrieve
     │               │          │
     ▼               │          │        ┌──────────┐
  Retrieved chunks ─→│          │ ──────→│ RETRIEVAL│ hit_rate, MRR, P@K
     │               │          │        │ EVALUATOR│
     ▼               │          │        └──────────┘
  LLM generation ──→│          │ t_generate
     │               │          │        ┌──────────┐
     ▼               └──────────┘ ──────→│ COST     │ tokens_in, tokens_out
  Answer ─────────────────────────────→│ TRACKER  │
     │                                   └──────────┘
     ▼
  ┌──────────────┐
  │ GROUNDEDNESS │  claim extraction + verification
  │ EVALUATOR    │
  └──────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import time
import math
import textwrap
import warnings
from collections import Counter
from dataclasses import dataclass, field

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 3

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English."""
    return max(1, len(text) // 4)


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Build the RAG Pipeline

## 8. Knowledge Base — Company Documentation

In [ ]:
CHUNKS = [
    {"id": "C01", "topic": "remote_work",
     "text": "Remote work is permitted up to 4 days per week. All employees must attend one mandatory in-office day for team collaboration. Remote workers receive a $1,200 annual stipend for home office equipment. Core hours are 10 AM to 3 PM in the employee's local timezone."},

    {"id": "C02", "topic": "remote_work",
     "text": "Fully remote positions are available for engineering and product roles. These require quarterly in-person team offsites (company-funded). Fully remote employees are eligible for $300/month coworking space reimbursement."},

    {"id": "C03", "topic": "security",
     "text": "All employees must enable multi-factor authentication (MFA) on all company accounts. Hardware security keys are preferred; TOTP apps are acceptable. SMS-based 2FA is not allowed due to SIM-swap risk. MFA enrollment must be completed within 24 hours of account creation."},

    {"id": "C04", "topic": "security",
     "text": "Passwords must be at least 16 characters using a passphrase format. Password rotation is not required per NIST 800-63B guidelines. Password managers are mandatory — the company provides 1Password licenses. Breached password detection runs weekly."},

    {"id": "C05", "topic": "security",
     "text": "USB drives and external storage devices are prohibited on company networks. All file sharing must use approved cloud platforms: Google Drive, SharePoint, or the internal file server. DLP scanning is active on all outbound channels including email and Slack."},

    {"id": "C06", "topic": "benefits",
     "text": "Health insurance covers medical, dental, and vision for employees and dependents. The company pays 90% of premiums for employees and 70% for dependents. Three plan options: HMO, PPO, and HDHP with HSA. Open enrollment is in November each year."},

    {"id": "C07", "topic": "benefits",
     "text": "The company matches 401(k) contributions up to 6% of salary with immediate vesting. RSU grants are awarded annually based on performance level. The stock purchase plan (ESPP) offers a 15% discount on company shares with semi-annual purchase windows."},

    {"id": "C08", "topic": "benefits",
     "text": "Parental leave policy: 16 weeks fully paid for all parents regardless of gender or birth/adoption status. Additional 4 weeks at 60% pay available upon request. Gradual return-to-work program available with reduced hours for 4 weeks after leave."},

    {"id": "C09", "topic": "pto",
     "text": "The company offers unlimited PTO with a minimum of 15 days encouraged per year. Manager approval is required for absences longer than 5 consecutive business days. Company-wide shutdown periods: last week of December and July 4th week. Unused PTO does not accrue or pay out."},

    {"id": "C10", "topic": "onboarding",
     "text": "New employee onboarding spans 4 weeks. Week 1: orientation, IT setup, compliance training. Week 2: product deep-dives and architecture overview. Weeks 3-4: paired assignments with a mentor. A 30-day check-in and 90-day review with the manager are mandatory."},

    {"id": "C11", "topic": "performance",
     "text": "Performance reviews occur semi-annually in June and December. Self-assessments are due 2 weeks before the review meeting. Ratings scale: 1 (Below Expectations) to 5 (Exceptional). Calibration sessions ensure consistency across teams. Promotion decisions are made during Q1 and Q3 calibration cycles."},

    {"id": "C12", "topic": "performance",
     "text": "Merit increases are tied to performance ratings: Below Expectations receives 0%, Meets gets 3-4%, Exceeds gets 5-8%, and Exceptional gets 9-15%. Spot bonuses up to $5,000 can be awarded by VPs at any time for extraordinary contributions."},

    {"id": "C13", "topic": "incidents",
     "text": "Incident severity levels: SEV1 (complete outage, page immediately), SEV2 (significant degradation, respond within 15 minutes), SEV3 (minor impact, respond within 1 hour), SEV4 (cosmetic, next business day). All SEV1 and SEV2 incidents require a post-incident review within 5 business days."},

    {"id": "C14", "topic": "incidents",
     "text": "During a SEV1 incident: the on-call engineer pages the incident commander within 5 minutes. A war room is opened in Slack (#incident-active). Status page is updated within 15 minutes. Customer communication goes out within 30 minutes. The incident commander owns all external communication."},

    {"id": "C15", "topic": "api",
     "text": "API rate limits: Free tier — 100 requests/minute. Pro tier — 2,000 requests/minute with burst to 3,000. Enterprise tier — 10,000 requests/minute. Rate limit headers: X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Reset. HTTP 429 returned when exceeded with Retry-After header."},

    {"id": "C16", "topic": "api",
     "text": "API authentication uses OAuth 2.0 with PKCE for user-facing apps and API keys for server-to-server integrations. API keys rotate every 90 days automatically. All API traffic must use TLS 1.2 or higher. mTLS is available for Enterprise tier customers."},

    {"id": "C17", "topic": "compliance",
     "text": "Data privacy compliance: GDPR for EU customers, CCPA for California residents, LGPD for Brazil. Customer data residency options: US (Virginia), EU (Ireland), APAC (Singapore). Data retention: 5 years for financial records, 1 year for activity logs. Automated DSAR fulfillment averages 2 business days."},

    {"id": "C18", "topic": "compliance",
     "text": "Security certifications: SOC 2 Type II (annual), ISO 27001 (certified), HIPAA BAA available for healthcare customers. Penetration testing conducted quarterly by independent third parties. Bug bounty program: up to $10,000 for critical vulnerabilities."},
]

print(f"Knowledge base: {len(CHUNKS)} chunks")
print(f"Topics: {dict(Counter(c['topic'] for c in CHUNKS))}")
print(f"Avg chunk length: {sum(len(c['text'].split()) for c in CHUNKS) // len(CHUNKS)} words")

In [ ]:
# Index in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("kb")
except Exception:
    pass

collection = chroma_client.create_collection(name="kb", metadata={"hnsw:space": "cosine"})

texts = [c["text"] for c in CHUNKS]
metas = [{"chunk_id": c["id"], "topic": c["topic"]} for c in CHUNKS]
ids = [c["id"] for c in CHUNKS]
vecs = embeddings.embed_documents(texts)
collection.add(documents=texts, embeddings=vecs, metadatas=metas, ids=ids)

print(f"Indexed {collection.count()} chunks")

## 9. RAG Pipeline with Instrumentation

In [ ]:
@dataclass
class RAGTrace:
    """Records everything about a single RAG execution."""
    question: str = ""
    # Retrieval
    retrieved_ids: list = field(default_factory=list)
    retrieved_scores: list = field(default_factory=list)
    retrieved_texts: list = field(default_factory=list)
    # Generation
    answer: str = ""
    # Cost
    prompt_tokens: int = 0
    output_tokens: int = 0
    context_tokens: int = 0
    total_tokens: int = 0
    # Latency (seconds)
    t_embed: float = 0.0
    t_retrieve: float = 0.0
    t_generate: float = 0.0
    t_total: float = 0.0


ANSWER_SYSTEM = """Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so.
Do not add information beyond what the context provides.
Cite chunks by ID: [C01], [C15], etc."""


def rag_query(question: str, top_k: int = TOP_K) -> RAGTrace:
    """Execute a RAG query with full instrumentation."""
    trace = RAGTrace(question=question)
    t_start = time.time()

    # 1. Embed query
    t0 = time.time()
    qv = embeddings.embed_query(question)
    trace.t_embed = time.time() - t0

    # 2. Retrieve
    t0 = time.time()
    results = collection.query(query_embeddings=[qv], n_results=top_k)
    trace.t_retrieve = time.time() - t0

    for i in range(len(results["documents"][0])):
        trace.retrieved_ids.append(results["metadatas"][0][i]["chunk_id"])
        trace.retrieved_scores.append(round(1 - results["distances"][0][i], 4))
        trace.retrieved_texts.append(results["documents"][0][i])

    # 3. Build prompt
    context = "\n\n".join(
        f"[{trace.retrieved_ids[i]}]: {trace.retrieved_texts[i]}"
        for i in range(len(trace.retrieved_ids))
    )
    prompt = f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nAnswer:"

    # Track token usage
    trace.context_tokens = estimate_tokens(context)
    trace.prompt_tokens = estimate_tokens(ANSWER_SYSTEM + prompt)

    # 4. Generate
    t0 = time.time()
    answer = ask(prompt, system=ANSWER_SYSTEM, temperature=0.1)
    trace.t_generate = time.time() - t0

    trace.answer = answer
    trace.output_tokens = estimate_tokens(answer)
    trace.total_tokens = trace.prompt_tokens + trace.output_tokens
    trace.t_total = time.time() - t_start

    return trace


# Test
test_trace = rag_query("What is the remote work policy?")
print(f"Question: {test_trace.question}")
print(f"Retrieved: {test_trace.retrieved_ids} (scores: {test_trace.retrieved_scores})")
print(f"Answer: {test_trace.answer[:120]}...")
print(f"Tokens: prompt={test_trace.prompt_tokens}, output={test_trace.output_tokens}, total={test_trace.total_tokens}")
print(f"Latency: embed={test_trace.t_embed:.3f}s, retrieve={test_trace.t_retrieve:.3f}s, "
      f"generate={test_trace.t_generate:.1f}s, total={test_trace.t_total:.1f}s")

---

# Part B — Evaluation Dataset

## 10. Labeled QA Pairs with Expected Chunks

A proper evaluation dataset specifies:
- The **question**
- Which **chunks** should be retrieved (ground truth retrieval)
- A **reference answer** (ground truth for answer quality)
- **Key facts** that must appear in the answer

In [ ]:
EVAL_SET = [
    {"id": "Q01",
     "question": "How many days per week can employees work remotely?",
     "expected_chunks": ["C01"],
     "key_facts": ["4 days per week", "one mandatory in-office day"],
     "difficulty": "easy"},

    {"id": "Q02",
     "question": "What are the password requirements and is rotation required?",
     "expected_chunks": ["C04"],
     "key_facts": ["16 characters", "rotation is not required", "NIST 800-63B"],
     "difficulty": "easy"},

    {"id": "Q03",
     "question": "What types of multi-factor authentication are allowed?",
     "expected_chunks": ["C03"],
     "key_facts": ["hardware security keys", "TOTP", "SMS not allowed"],
     "difficulty": "easy"},

    {"id": "Q04",
     "question": "How long is parental leave and does it apply to all parents?",
     "expected_chunks": ["C08"],
     "key_facts": ["16 weeks", "fully paid", "regardless of gender"],
     "difficulty": "easy"},

    {"id": "Q05",
     "question": "What is the company's 401k matching policy and vesting schedule?",
     "expected_chunks": ["C07"],
     "key_facts": ["6% of salary", "immediate vesting"],
     "difficulty": "medium"},

    {"id": "Q06",
     "question": "What happens during a SEV1 incident — who gets paged and what's the communication timeline?",
     "expected_chunks": ["C13", "C14"],
     "key_facts": ["page immediately", "incident commander", "5 minutes", "status page within 15 minutes", "customer communication within 30 minutes"],
     "difficulty": "hard"},

    {"id": "Q07",
     "question": "What are the API rate limits for each tier?",
     "expected_chunks": ["C15"],
     "key_facts": ["100 requests/minute", "2,000", "10,000", "429"],
     "difficulty": "medium"},

    {"id": "Q08",
     "question": "What data privacy regulations does the company comply with and where is data stored?",
     "expected_chunks": ["C17"],
     "key_facts": ["GDPR", "CCPA", "LGPD", "Virginia", "Ireland", "Singapore"],
     "difficulty": "medium"},

    {"id": "Q09",
     "question": "What is the full remote work equipment support — stipend, coworking, and eligibility?",
     "expected_chunks": ["C01", "C02"],
     "key_facts": ["$1,200 annual stipend", "$300/month coworking", "engineering and product roles"],
     "difficulty": "hard"},

    {"id": "Q10",
     "question": "How does the performance review cycle work — timing, ratings, and merit increases?",
     "expected_chunks": ["C11", "C12"],
     "key_facts": ["semi-annually", "June and December", "1 to 5", "3-4%", "9-15%"],
     "difficulty": "hard"},
]

print(f"Evaluation set: {len(EVAL_SET)} questions")
print(f"Difficulty: {dict(Counter(q['difficulty'] for q in EVAL_SET))}")
print(f"Multi-chunk questions: {sum(1 for q in EVAL_SET if len(q['expected_chunks']) > 1)}")

## 11. Run All Evaluation Queries

In [ ]:
print("Running all evaluation queries...\n")

traces = []
for i, item in enumerate(EVAL_SET, 1):
    trace = rag_query(item["question"])
    traces.append(trace)
    print(f"  [{i}/{len(EVAL_SET)}] {item['id']} retrieved={trace.retrieved_ids}  "
          f"tokens={trace.total_tokens}  time={trace.t_total:.1f}s")

print(f"\nAll {len(traces)} queries complete.")

---

# Part C — Pillar 1: Retrieval Quality

## 12. Retrieval Metrics

| Metric | What It Measures | Formula |
|--------|-----------------|--------|
| **Hit Rate** | Was at least one expected chunk retrieved? | `1 if any expected chunk in retrieved else 0` |
| **MRR** | How high is the first relevant chunk ranked? | `1 / rank_of_first_relevant` |
| **Precision@K** | What fraction of retrieved chunks are relevant? | `relevant_retrieved / K` |
| **Recall@K** | What fraction of all relevant chunks were retrieved? | `relevant_retrieved / total_relevant` |

In [ ]:
def compute_retrieval_metrics(retrieved_ids: list, expected_ids: list) -> dict:
    """Compute retrieval quality metrics for a single query."""
    expected_set = set(expected_ids)
    k = len(retrieved_ids)

    # Hit Rate: is any expected chunk in the retrieved set?
    hit = int(bool(expected_set & set(retrieved_ids)))

    # MRR: reciprocal rank of first relevant chunk
    mrr = 0.0
    for rank, rid in enumerate(retrieved_ids, 1):
        if rid in expected_set:
            mrr = 1.0 / rank
            break

    # Precision@K: fraction of retrieved that are relevant
    relevant_retrieved = sum(1 for r in retrieved_ids if r in expected_set)
    precision = relevant_retrieved / k if k > 0 else 0.0

    # Recall@K: fraction of relevant that were retrieved
    recall = relevant_retrieved / len(expected_set) if expected_set else 0.0

    return {
        "hit": hit,
        "mrr": round(mrr, 4),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "relevant_retrieved": relevant_retrieved,
    }


# Compute for all queries
retrieval_results = []
for item, trace in zip(EVAL_SET, traces):
    metrics = compute_retrieval_metrics(trace.retrieved_ids, item["expected_chunks"])
    metrics["qid"] = item["id"]
    metrics["difficulty"] = item["difficulty"]
    metrics["retrieved"] = trace.retrieved_ids
    metrics["expected"] = item["expected_chunks"]
    retrieval_results.append(metrics)

# Print per-query results
print("RETRIEVAL QUALITY (per query)")
print("=" * 95)
print(f"{'QID':<5} {'Diff':<7} {'Hit':<5} {'MRR':<6} {'P@K':<6} {'R@K':<6} {'Retrieved':<20} {'Expected'}")
print("-" * 95)

for r in retrieval_results:
    print(f"{r['qid']:<5} {r['difficulty']:<7} {r['hit']:<5} {r['mrr']:<6.2f} "
          f"{r['precision']:<6.2f} {r['recall']:<6.2f} "
          f"{str(r['retrieved']):<20} {r['expected']}")

# Averages
n = len(retrieval_results)
print(f"\n  AVERAGES:")
print(f"    Hit Rate:     {sum(r['hit'] for r in retrieval_results) / n:.2f}")
print(f"    MRR:          {sum(r['mrr'] for r in retrieval_results) / n:.3f}")
print(f"    Precision@{TOP_K}: {sum(r['precision'] for r in retrieval_results) / n:.3f}")
print(f"    Recall@{TOP_K}:    {sum(r['recall'] for r in retrieval_results) / n:.3f}")

In [ ]:
# Breakdown by difficulty
print("RETRIEVAL QUALITY BY DIFFICULTY")
print("=" * 55)

for diff in ["easy", "medium", "hard"]:
    items = [r for r in retrieval_results if r["difficulty"] == diff]
    if not items:
        continue
    nd = len(items)
    print(f"  {diff.upper()} ({nd} questions):")
    print(f"    Hit Rate: {sum(r['hit'] for r in items) / nd:.2f}")
    print(f"    MRR:      {sum(r['mrr'] for r in items) / nd:.3f}")
    print(f"    Recall:   {sum(r['recall'] for r in items) / nd:.3f}")

# Identify retrieval failures
failures = [r for r in retrieval_results if r["hit"] == 0]
low_recall = [r for r in retrieval_results if r["recall"] < 1.0 and r["hit"] == 1]
print(f"\n  Complete misses: {len(failures)}")
print(f"  Partial recall (hit but not all expected chunks): {len(low_recall)}")
for r in low_recall:
    print(f"    {r['qid']}: got {r['retrieved']}, needed {r['expected']}")

---

# Part D — Pillar 2: Groundedness

## 13. Claim-Level Groundedness Evaluation

Groundedness = Is every claim in the generated answer **actually supported** by the retrieved chunks?

Steps:
1. Extract individual claims from the answer
2. For each claim, check if it's supported by the retrieved context
3. Compute: `groundedness = supported_claims / total_claims`

In [ ]:
EXTRACT_SYSTEM = """Extract specific factual claims from the answer.
Each claim should be a single verifiable statement.
Skip vague filler like 'the policy covers this topic'."""

EXTRACT_PROMPT = """ANSWER: {answer}

Extract factual claims as a JSON list of strings.
Return ONLY valid JSON: ["claim 1", "claim 2", ...]"""

VERIFY_SYSTEM = """Determine if a claim is SUPPORTED by the evidence.
SUPPORTED: The evidence directly states or clearly implies this fact.
NOT_SUPPORTED: The evidence does not contain this information, or contradicts it."""

VERIFY_PROMPT = """CLAIM: {claim}

EVIDENCE:
{evidence}

Is this claim supported by the evidence? Return ONLY JSON:
{{"verdict": "SUPPORTED" or "NOT_SUPPORTED",
  "reasoning": "brief explanation"}}"""


def evaluate_groundedness(answer: str, context_texts: list) -> dict:
    """Evaluate groundedness of an answer against retrieved context."""
    # Step 1: Extract claims
    claims_raw = parse_json(ask(
        EXTRACT_PROMPT.format(answer=answer[:500]),
        system=EXTRACT_SYSTEM, temperature=0.0,
    ))
    if not isinstance(claims_raw, list) or not claims_raw:
        return {"claims": [], "supported": 0, "total": 0, "score": 0.0, "verdicts": []}

    claims = [str(c) for c in claims_raw[:8]]  # Cap at 8 claims
    evidence = "\n\n".join(context_texts)

    # Step 2: Verify each claim
    verdicts = []
    for claim in claims:
        result = parse_json(ask(
            VERIFY_PROMPT.format(claim=claim, evidence=evidence[:1500]),
            system=VERIFY_SYSTEM, temperature=0.0,
        ))
        verdict = result.get("verdict", "NOT_SUPPORTED") if result else "NOT_SUPPORTED"
        verdicts.append({
            "claim": claim,
            "verdict": verdict,
            "reasoning": result.get("reasoning", "") if result else "",
        })

    supported = sum(1 for v in verdicts if v["verdict"] == "SUPPORTED")
    total = len(verdicts)
    score = supported / total if total > 0 else 0.0

    return {
        "claims": claims,
        "supported": supported,
        "total": total,
        "score": round(score, 3),
        "verdicts": verdicts,
    }


print("Groundedness evaluator ready")

In [ ]:
print("GROUNDEDNESS EVALUATION")
print("=" * 90)

groundedness_results = []

for i, (item, trace) in enumerate(zip(EVAL_SET, traces), 1):
    g = evaluate_groundedness(trace.answer, trace.retrieved_texts)
    g["qid"] = item["id"]
    g["difficulty"] = item["difficulty"]
    groundedness_results.append(g)

    status = "PASS" if g["score"] >= 0.8 else "WARN" if g["score"] >= 0.5 else "FAIL"
    print(f"  [{i}/{len(EVAL_SET)}] {item['id']} — {g['supported']}/{g['total']} claims supported "
          f"({g['score']:.0%}) [{status}]")

# Summary
avg_ground = sum(g["score"] for g in groundedness_results) / len(groundedness_results)
perfect = sum(1 for g in groundedness_results if g["score"] == 1.0)
print(f"\n  Average groundedness: {avg_ground:.1%}")
print(f"  Perfectly grounded:   {perfect}/{len(groundedness_results)}")

In [ ]:
# Show detailed claim-level results for one query
print("DETAILED GROUNDEDNESS — sample query")
print("=" * 80)

# Pick a query with interesting results (not perfect score)
detail_idx = next(
    (i for i, g in enumerate(groundedness_results) if 0 < g["score"] < 1.0),
    0,  # fallback to first
)
g = groundedness_results[detail_idx]
item = EVAL_SET[detail_idx]
trace = traces[detail_idx]

print(f"Q: {item['question']}")
print(f"Answer: {trace.answer[:200]}...")
print(f"Retrieved chunks: {trace.retrieved_ids}")
print(f"\nClaim-level verdicts ({g['score']:.0%} grounded):")

for v in g["verdicts"]:
    icon = "+" if v["verdict"] == "SUPPORTED" else "-"
    print(f"  [{icon}] {v['verdict']}: {v['claim'][:70]}")
    if v["reasoning"]:
        print(f"      Reason: {v['reasoning'][:60]}")

## 14. Key Fact Coverage

Separate from groundedness (are claims supported?), **key fact coverage** checks whether the **expected facts** actually appear in the answer.

In [ ]:
print("KEY FACT COVERAGE")
print("=" * 90)

factual_results = []

for item, trace in zip(EVAL_SET, traces):
    answer_lower = trace.answer.lower()
    found = []
    missed = []
    for fact in item["key_facts"]:
        if fact.lower() in answer_lower:
            found.append(fact)
        else:
            missed.append(fact)

    coverage = len(found) / len(item["key_facts"]) if item["key_facts"] else 0
    factual_results.append({
        "qid": item["id"],
        "coverage": round(coverage, 3),
        "found": found,
        "missed": missed,
    })

    status = f"{len(found)}/{len(item['key_facts'])}"
    icon = "+" if coverage == 1.0 else "~" if coverage >= 0.5 else "-"
    missed_str = f"  missing: {missed[:3]}" if missed else ""
    print(f"  [{icon}] {item['id']} facts={status} ({coverage:.0%}){missed_str}")

avg_coverage = sum(r["coverage"] for r in factual_results) / len(factual_results)
print(f"\n  Average fact coverage: {avg_coverage:.1%}")

---

# Part E — Pillar 3: Cost Analysis

## 15. Token Budget Breakdown

Even with local LLMs (zero dollar cost), understanding token usage is critical for:
- Estimating **production costs** when migrating to a hosted API
- Staying within **context window limits**
- Optimizing **prompt efficiency**

In [ ]:
print("TOKEN USAGE ANALYSIS")
print("=" * 80)

# Per-query breakdown
print(f"{'QID':<5} {'Context':<10} {'Prompt':<10} {'Output':<10} {'Total':<10}")
print("-" * 45)

for item, trace in zip(EVAL_SET, traces):
    print(f"{item['id']:<5} {trace.context_tokens:<10} {trace.prompt_tokens:<10} "
          f"{trace.output_tokens:<10} {trace.total_tokens:<10}")

# Aggregate stats
total_context = sum(t.context_tokens for t in traces)
total_prompt = sum(t.prompt_tokens for t in traces)
total_output = sum(t.output_tokens for t in traces)
total_all = sum(t.total_tokens for t in traces)
n = len(traces)

print(f"\n  TOTALS (across {n} queries):")
print(f"    Context tokens: {total_context:,} (avg {total_context // n:,}/query)")
print(f"    Prompt tokens:  {total_prompt:,} (avg {total_prompt // n:,}/query)")
print(f"    Output tokens:  {total_output:,} (avg {total_output // n:,}/query)")
print(f"    Total tokens:   {total_all:,} (avg {total_all // n:,}/query)")

# Context is typically 70-85% of total tokens
context_pct = total_context / total_prompt * 100
print(f"\n  Context is {context_pct:.0f}% of prompt tokens.")
print(f"  Reducing chunk count or chunk size is the most effective cost optimization.")

In [ ]:
# Cost projection for hosted APIs
PRICING = {
    "GPT-4o": {"input": 2.50, "output": 10.00},
    "GPT-4o-mini": {"input": 0.15, "output": 0.60},
    "Claude Sonnet": {"input": 3.00, "output": 15.00},
    "Claude Haiku": {"input": 0.25, "output": 1.25},
    "Ollama (local)": {"input": 0.00, "output": 0.00},
}

avg_in = total_prompt / n
avg_out = total_output / n

print("COST PROJECTION — if these queries ran on hosted APIs")
print("=" * 70)
print(f"Based on avg {avg_in:.0f} input + {avg_out:.0f} output tokens per query\n")
print(f"{'Model':<18} {'$/1K queries':<14} {'$/10K queries':<14} {'$/100K queries'}")
print("-" * 60)

for model, prices in PRICING.items():
    per_query = (avg_in / 1_000_000 * prices["input"]) + (avg_out / 1_000_000 * prices["output"])
    cost_1k = per_query * 1_000
    cost_10k = per_query * 10_000
    cost_100k = per_query * 100_000
    print(f"  {model:<18} ${cost_1k:<13.2f} ${cost_10k:<13.2f} ${cost_100k:,.2f}")

print(f"\n  With Ollama: $0 regardless of volume. Trade-off: latency & hardware.")

---

# Part F — Pillar 4: Latency Profiling

## 16. Latency Breakdown by Stage

In [ ]:
print("LATENCY ANALYSIS")
print("=" * 80)

# Per-query latency
print(f"{'QID':<5} {'Embed(s)':<10} {'Retrieve(s)':<12} {'Generate(s)':<12} {'Total(s)':<10}")
print("-" * 49)

for item, trace in zip(EVAL_SET, traces):
    print(f"{item['id']:<5} {trace.t_embed:<10.3f} {trace.t_retrieve:<12.3f} "
          f"{trace.t_generate:<12.1f} {trace.t_total:<10.1f}")

# Stage averages
avg_embed = sum(t.t_embed for t in traces) / n
avg_retrieve = sum(t.t_retrieve for t in traces) / n
avg_generate = sum(t.t_generate for t in traces) / n
avg_total = sum(t.t_total for t in traces) / n

print(f"\n  AVERAGES:")
print(f"    Embed:    {avg_embed:.3f}s ({avg_embed / avg_total * 100:.1f}% of total)")
print(f"    Retrieve: {avg_retrieve:.3f}s ({avg_retrieve / avg_total * 100:.1f}% of total)")
print(f"    Generate: {avg_generate:.1f}s ({avg_generate / avg_total * 100:.1f}% of total)")
print(f"    Total:    {avg_total:.1f}s")

# Percentiles
totals = sorted(t.t_total for t in traces)
p50 = totals[len(totals) // 2]
p90 = totals[int(len(totals) * 0.9)]
p99 = totals[-1]  # max for small datasets
print(f"\n  PERCENTILES:")
print(f"    P50: {p50:.1f}s")
print(f"    P90: {p90:.1f}s")
print(f"    Max: {p99:.1f}s")
print(f"\n  LLM generation dominates latency ({avg_generate / avg_total * 100:.0f}%).")
print(f"  Embed + retrieve combined is typically <5% of total time.")

In [ ]:
# Tokens per second (generation throughput)
print("GENERATION THROUGHPUT")
print("=" * 50)

throughputs = []
for item, trace in zip(EVAL_SET, traces):
    if trace.t_generate > 0:
        tps = trace.output_tokens / trace.t_generate
        throughputs.append(tps)
        print(f"  {item['id']}: {trace.output_tokens} tokens / {trace.t_generate:.1f}s = {tps:.1f} tok/s")

if throughputs:
    print(f"\n  Average throughput: {sum(throughputs) / len(throughputs):.1f} tokens/s")
    print(f"  This is the output generation speed of the local LLM.")
    print(f"  Hosted APIs typically achieve 50-100+ tokens/s.")

---

# Part G — RAG Scorecard

## 17. Unified Evaluation Scorecard

In [ ]:
def grade(value: float, thresholds: tuple) -> str:
    """Assign a letter grade based on thresholds: (A, B, C cutoffs)."""
    if value >= thresholds[0]:
        return "A"
    elif value >= thresholds[1]:
        return "B"
    elif value >= thresholds[2]:
        return "C"
    else:
        return "F"


# Compute scorecard metrics
hit_rate = sum(r["hit"] for r in retrieval_results) / len(retrieval_results)
avg_mrr = sum(r["mrr"] for r in retrieval_results) / len(retrieval_results)
avg_precision = sum(r["precision"] for r in retrieval_results) / len(retrieval_results)
avg_recall = sum(r["recall"] for r in retrieval_results) / len(retrieval_results)
avg_groundedness = sum(g["score"] for g in groundedness_results) / len(groundedness_results)
avg_fact_coverage = sum(r["coverage"] for r in factual_results) / len(factual_results)
avg_tokens_query = total_all / n
avg_latency = avg_total

print("╔══════════════════════════════════════════════════════════════╗")
print("║              RAG PIPELINE EVALUATION SCORECARD              ║")
print("╠══════════════════════════════════════════════════════════════╣")
print("║                                                             ║")
print("║  PILLAR 1: RETRIEVAL QUALITY                                ║")
print(f"║    Hit Rate:      {hit_rate:.0%}    [{grade(hit_rate, (0.9, 0.7, 0.5))}]{'':>27}║")
print(f"║    MRR:           {avg_mrr:.3f}  [{grade(avg_mrr, (0.9, 0.7, 0.5))}]{'':>27}║")
print(f"║    Precision@{TOP_K}:  {avg_precision:.3f}  [{grade(avg_precision, (0.7, 0.5, 0.3))}]{'':>27}║")
print(f"║    Recall@{TOP_K}:     {avg_recall:.3f}  [{grade(avg_recall, (0.9, 0.7, 0.5))}]{'':>27}║")
print("║                                                             ║")
print("║  PILLAR 2: GROUNDEDNESS                                     ║")
print(f"║    Groundedness:  {avg_groundedness:.0%}    [{grade(avg_groundedness, (0.9, 0.7, 0.5))}]{'':>27}║")
print(f"║    Fact Coverage: {avg_fact_coverage:.0%}    [{grade(avg_fact_coverage, (0.9, 0.7, 0.5))}]{'':>27}║")
print("║                                                             ║")
print("║  PILLAR 3: COST                                             ║")
print(f"║    Avg tokens/query: {avg_tokens_query:,.0f}{'':>34}║")
print(f"║    Context share:    {context_pct:.0f}% of prompt{'':>27}║")
print(f"║    Local LLM cost:   $0.00 / query{'':>22}║")
print("║                                                             ║")
print("║  PILLAR 4: LATENCY                                          ║")
print(f"║    Avg total:   {avg_latency:.1f}s{'':>39}║")
print(f"║    P50:         {p50:.1f}s{'':>39}║")
print(f"║    P90:         {p90:.1f}s{'':>39}║")
print(f"║    Bottleneck:  LLM generation ({avg_generate / avg_total * 100:.0f}%){'':>22}║")
print("║                                                             ║")
print("╚══════════════════════════════════════════════════════════════╝")

## 18. Per-Query Scorecard

In [ ]:
print("PER-QUERY SCORECARD")
print("=" * 100)
print(f"{'QID':<5} {'Diff':<7} {'Hit':<5} {'MRR':<6} {'Recall':<7} {'Ground':<8} {'Facts':<7} "
      f"{'Tokens':<8} {'Time(s)':<8} {'Overall'}")
print("-" * 100)

overall_scores = []

for i, (item, trace) in enumerate(zip(EVAL_SET, traces)):
    r = retrieval_results[i]
    g = groundedness_results[i]
    f = factual_results[i]

    # Composite score: weighted average of retrieval + groundedness + fact coverage
    composite = (r["recall"] * 0.3 + g["score"] * 0.4 + f["coverage"] * 0.3)
    overall_scores.append(composite)

    status = "PASS" if composite >= 0.7 else "WARN" if composite >= 0.5 else "FAIL"

    print(f"{item['id']:<5} {item['difficulty']:<7} {r['hit']:<5} {r['mrr']:<6.2f} {r['recall']:<7.2f} "
          f"{g['score']:<8.0%} {f['coverage']:<7.0%} {trace.total_tokens:<8} {trace.t_total:<8.1f} [{status}]")

avg_overall = sum(overall_scores) / len(overall_scores)
pass_rate = sum(1 for s in overall_scores if s >= 0.7) / len(overall_scores)
print(f"\n  Overall composite score: {avg_overall:.1%}")
print(f"  Pass rate (>=70%):       {pass_rate:.0%}")

---

# Part H — Analysis & Improvement Ideas

## 19. Failure Analysis

In [ ]:
# Find the weakest queries
weakest = sorted(enumerate(overall_scores), key=lambda x: x[1])[:3]

print("FAILURE ANALYSIS — weakest queries")
print("=" * 90)

for idx, score in weakest:
    item = EVAL_SET[idx]
    trace = traces[idx]
    r = retrieval_results[idx]
    g = groundedness_results[idx]
    f = factual_results[idx]

    print(f"\n  {item['id']} (composite={score:.0%}, difficulty={item['difficulty']})")
    print(f"  Q: {item['question']}")

    # Diagnose root cause
    issues = []
    if r["recall"] < 1.0:
        issues.append(f"Retrieval: recall={r['recall']:.0%} — missed chunks {set(item['expected_chunks']) - set(trace.retrieved_ids)}")
    if g["score"] < 0.8:
        unsupported = [v["claim"][:50] for v in g["verdicts"] if v["verdict"] != "SUPPORTED"]
        issues.append(f"Groundedness: {g['score']:.0%} — unsupported claims: {unsupported[:2]}")
    if f["missed"]:
        issues.append(f"Missing facts: {f['missed'][:3]}")

    if not issues:
        issues.append("No clear failure mode — may be borderline scoring")

    for issue in issues:
        print(f"    → {issue}")

## 20. Common Mistakes in RAG Evaluation

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Only checking if the answer "looks right" | Vibes-based evaluation misses hallucinations | Claim-level groundedness verification |
| Not having a labeled evaluation set | Can't measure retrieval quality without ground-truth chunks | Create QA pairs with expected chunks and key facts |
| Ignoring retrieval and only evaluating answers | Bad retrieval → bad answers; fix retrieval first | Evaluate retrieval quality separately |
| Using accuracy as the only metric | 90% accurate but 50% grounded is dangerous | Multi-pillar: retrieval + groundedness + cost + latency |
| Not tracking costs | Fine in development, catastrophic in production | Estimate tokens per query from the start |
| Measuring average latency only | P50 can be fine while P99 is terrible | Track percentiles: P50, P90, P99 |
| Evaluating once and shipping | Model drift, data changes → metrics degrade | Continuous evaluation in production |

## 21. Production Improvement Ideas

1. **Increase top_k for hard questions** — multi-chunk questions need more retrieved context
2. **Chunk reranking** — use a cross-encoder to rerank initial results and improve precision
3. **Context compression** — summarize retrieved chunks to reduce token cost before generation
4. **Streaming responses** — start sending tokens while generating to improve perceived latency
5. **Retrieval caching** — cache embeddings and results for repeated queries
6. **Automated regression testing** — run the eval set on every pipeline change
7. **Production monitoring** — track groundedness, latency, and cost in real-time dashboards

## 22. Exercises

### Exercise 1
Implement **answer relevance scoring**: use an LLM to judge whether the answer actually addresses the question asked (not just whether it's grounded). A perfectly grounded answer can still be irrelevant if the wrong chunks were retrieved.

### Exercise 2
Add **chunk-level attribution**: for each sentence in the answer, identify which specific chunk it came from. Compute an "attribution clarity" score — answers should clearly map to their source chunks.

### Exercise 3
Build a **retrieval improvement loop**: for queries with low recall, generate synthetic variations of the question and check if alternative phrasings retrieve the correct chunks. Use this to identify embedding blind spots.

### Mini Challenge
Implement **automated regression detection**: compare two versions of the pipeline (e.g., different chunk sizes, different top_k) on the full eval set. Produce a diff report showing which queries improved, which degraded, and whether the overall scorecard changed.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(CHUNKS)} chunks across {len(set(c['topic'] for c in CHUNKS))} topics")
print(f"Evaluation set:  {len(EVAL_SET)} questions ({dict(Counter(q['difficulty'] for q in EVAL_SET))})")
print()
print(f"RETRIEVAL QUALITY:")
print(f"  Hit Rate:   {hit_rate:.0%}")
print(f"  MRR:        {avg_mrr:.3f}")
print(f"  Recall@{TOP_K}:  {avg_recall:.3f}")
print()
print(f"GROUNDEDNESS:")
print(f"  Avg score:      {avg_groundedness:.0%}")
print(f"  Fact coverage:  {avg_fact_coverage:.0%}")
print()
print(f"COST:")
print(f"  Avg tokens/query: {avg_tokens_query:,.0f}")
print(f"  Local LLM:        $0.00")
print()
print(f"LATENCY:")
print(f"  Avg total:  {avg_latency:.1f}s")
print(f"  P90:        {p90:.1f}s")
print(f"  Bottleneck: LLM generation ({avg_generate / avg_total * 100:.0f}%)")
print()
print(f"COMPOSITE:")
print(f"  Overall score:  {avg_overall:.0%}")
print(f"  Pass rate:      {pass_rate:.0%}")
print()
print(f"Components built:")
print(f"  1. Instrumented RAG pipeline  — traces every stage")
print(f"  2. Retrieval evaluator        — hit rate, MRR, P@K, R@K")
print(f"  3. Groundedness evaluator     — claim extraction + verification")
print(f"  4. Key fact checker           — expected fact coverage")
print(f"  5. Cost analyzer              — token budget + API cost projection")
print(f"  6. Latency profiler           — per-stage timing + percentiles")
print(f"  7. RAG scorecard              — unified 4-pillar summary")
print(f"  8. Failure analyzer           — root cause for weakest queries")

## 23. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **RAG evaluation needs four pillars**: retrieval quality, groundedness, cost, and latency — not just "does the answer look right" |
| 2 | **Retrieval quality is upstream of everything** — bad retrieval guarantees bad answers regardless of LLM quality |
| 3 | **Groundedness ≠ correctness** — an answer can be grounded in wrong chunks; combine with retrieval metrics |
| 4 | **Claim-level verification** catches subtle hallucinations that answer-level checks miss |
| 5 | **Context tokens dominate cost** — reducing chunk count or size is the most effective cost optimization |
| 6 | **LLM generation dominates latency** — embed + retrieve is typically <5% of total pipeline time |
| 7 | **Track percentiles, not averages** — P50 can be fine while P90 is unacceptable |
| 8 | **A labeled eval set is essential** — without ground-truth chunks and facts, metrics are meaningless |
| 9 | **Multi-chunk questions are hardest** — they stress both retrieval (recall across chunks) and synthesis |
| 10 | **Evaluate continuously** — a pipeline that scores well today can degrade with data or model changes |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*